# Models Pipeline

This notebook is the single restartable entry point. Reusable implementation lives in `src/`, while completed annual refits and diagnostics are cached by model ID and signature.

## Model roster

The retained models cover conventional neural and non-neural benchmarks; the LightGBM 20/40/60/80/100 characteristic-breadth comparison; exact-calendar lagged LightGBM models; matched MLP and DeepSets models using 40 characteristics; supporting Core20 DeepSets models; and two validation-weighted hybrids. `NN2_20` and `NN2_40` use hidden-layer widths `[32, 16]`, `NN3_20` uses `[32, 16, 8]`, and `NN4_20` and `NN4_40` use `[32, 16, 8, 4]`. These are GKX-style benchmarks rather than exact replications because this project uses a different information set and sample. `MLP_40` remains the monthly-panel no-market-context control for `DEEPSET_40`; the separately implemented `NN2_40` permits a clean feedforward depth comparison with `NN4_40`. `DEEPSET_40_LAG1` adds current and exact one-month-lagged characteristics, while `DEEPSET_40_DYNAMIC` additionally includes rank velocities.

## Fixed design decisions

- The target is decimal next-month excess return, `ret_exc_lead1m`.
- The universe is USA stocks with valid PERMNO and size group micro/small/large/mega; nano stocks are excluded.
- Characteristics are ranked within the complete eligible monthly cross-section and mapped to `[-1, 1]` before target availability is inspected. Missing targets are masked from fitting and evaluation calculations but do not change the month-t ranking universe.
- Exact-calendar lags are joined by PERMNO. Missing prior months receive neutral values plus a zero availability flag.
- The rolling design uses 15 training years, 4 validation years, one untouched test year, and annual refits from 1999 through 2024.
- Validation MSE selects hyperparameters and early stopping. Test outcomes never affect fitting or model selection.
- Primary evaluation is pooled GKX OOS R-squared and the equal-weighted D10-D1 portfolio. Rank IC, calibration, robust R-squared, monotonicity, alternative portfolios, transaction costs, universe sensitivity, and an adverse missing-return stress are supporting diagnostics.
- Constant-forecast months hold cash; PERMNO is only a deterministic tie-break when a genuine signal exists.
- Each model/refit has an independent signature and completion marker. Compatible results load without retraining or overwriting.

## 1. Runtime and project setup

The notebook file remains local and is edited in VS Code. When the selected kernel is Google Colab, this cell mounts Google Drive and uses the Drive copy of the project for source code, data, checkpoints, predictions, and diagnostics. With a normal local VS Code kernel, it uses the local Windows project folder instead. This storage choice does not change model definitions or experiment signatures.

In [ ]:
import os
import sys
import importlib
from pathlib import Path

LOCAL_PROJECT_DIR = Path(r"C:\Users\sandh\OneDrive\Documents\Coding\FDS Project")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/FDS Project")

try:
    from google.colab import drive
except ImportError:
    RUNNING_IN_COLAB = False
    PROJECT_DIR = LOCAL_PROJECT_DIR
    RUNTIME = "Local VS Code kernel"
else:
    RUNNING_IN_COLAB = True
    # Safe under Run all: mounts Drive when needed and reuses an existing mount.
    drive.mount("/content/drive", force_remount=False)
    PROJECT_DIR = DRIVE_PROJECT_DIR
    RUNTIME = "Google Colab kernel in VS Code"

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        f"Project folder was not found: {PROJECT_DIR}\n"
        "If this is Colab, confirm that Drive is mounted and that the folder name matches exactly."
    )
if not (PROJECT_DIR / "src").is_dir():
    raise FileNotFoundError(f"The project src folder was not found under: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
project_path = str(PROJECT_DIR)
# Always give this project priority over stale or similarly named packages.
sys.path = [path for path in sys.path if path != project_path]
sys.path.insert(0, project_path)
for module_name in tuple(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]
importlib.invalidate_caches()

# Verify now, before any model configuration is evaluated.
import src
src_file = Path(src.__file__).resolve()
if PROJECT_DIR.resolve() not in src_file.parents:
    raise RuntimeError(f"Imported src from the wrong location: {src_file}")

print("Notebook file: local Models_Pipeline.ipynb")
print("Runtime:", RUNTIME)
print("Project files and outputs:", PROJECT_DIR)
print("Imported src from:", src_file)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Notebook file: local Models_Pipeline.ipynb
Runtime: Google Colab kernel in VS Code
Project files and outputs: /content/drive/MyDrive/Colab Notebooks/FDS Project
Imported src from: /content/drive/MyDrive/Colab Notebooks/FDS Project/src/__init__.py


## 2. Experiment configuration

Usually this is the only cell to edit. Use a new `experiment_id` after changing the universe, feature definition, target, preprocessing, or rolling schedule. The same experiment can safely be rerun or extended with new registered models.

In [ ]:
if 'PROJECT_DIR' not in globals():
    raise RuntimeError('Run the Runtime and project setup cell first, or use Run all.')

from src.config import ExperimentConfig
from src.models import MODEL_REGISTRY

DATA_PATH = PROJECT_DIR / 'jkp_USA_100chars_1980_2024.parquet'
OUTPUT_DIR = PROJECT_DIR / 'model_runs'
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Raw data file not found: {DATA_PATH}')

MODEL_ROSTER = (
    'LASSO_20', 'LGBM_20', 'XGBOOST_20', 'NN2_20', 'NN2_40', 'NN3_20', 'NN4_20', 'NN4_40',
    'LGBM_40', 'LGBM_60', 'LGBM_80', 'LGBM_100',
    'LGBM_20_LAG1', 'LGBM_20_LAG2',
    'LGBM_40_LAG1', 'LGBM_40_LAG2',
    'MLP_40', 'DEEPSET_40', 'DEEPSET_40_LAG1', 'DEEPSET_40_DYNAMIC',
    'DEEPSET_20', 'DEEPSET_20_LAG1', 'DEEPSET_20_DYNAMIC',
    'HYBRID_LGBM20_DEEPSET20', 'HYBRID_MLP40_DEEPSET40',
    'HYBRID_LGBM40_DEEPSET40', 'HYBRID_LGBM40_DEEPSET40_DYNAMIC',
)
# These hybrids reuse completed component checkpoints and estimate only
# a convex weight on each refit's four-year validation period.
SELECTED_MODELS = (
    'HYBRID_LGBM40_DEEPSET40',
    'HYBRID_LGBM40_DEEPSET40_DYNAMIC',
)

CONFIG = ExperimentConfig(
    experiment_id='core20_benchmarks_v1',
    project_dir=PROJECT_DIR,
    data_path=DATA_PATH,
    output_dir=OUTPUT_DIR,
    selected_models=SELECTED_MODELS,
    seed=42,
    use_gpu=True,
)
CONFIG.validate()

import torch
if CONFIG.use_gpu and not torch.cuda.is_available():
    raise RuntimeError('GPU requested but unavailable. In Colab choose Runtime > Change runtime type > T4 GPU, then reconnect.')
print('Torch device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Selected models:', list(CONFIG.selected_models))
print('Run directory:', CONFIG.run_dir)

Torch device: NVIDIA A100-SXM4-40GB
Selected models: ['HYBRID_LGBM40_DEEPSET40', 'HYBRID_LGBM40_DEEPSET40_DYNAMIC']
Run directory: /content/drive/MyDrive/Colab Notebooks/FDS Project/model_runs/core20_benchmarks_v1


## 3. Preflight checks

This verifies the Drive project files, model registry and deterministic feature-construction checks without loading the full panel. The runner loads and prepares the data once in the next section.

In [ ]:
import unittest
from src.self_checks import run_framework_self_checks

required_files = (
    DATA_PATH,
    PROJECT_DIR / 'src' / 'config.py',
    PROJECT_DIR / 'src' / 'models.py',
    PROJECT_DIR / 'src' / 'runner.py',
    PROJECT_DIR / 'src' / 'self_checks.py',
    PROJECT_DIR / 'tests' / 'test_pipeline.py',
)
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f'Required Drive project files are missing: {missing_files}')
unknown_models = sorted(set(CONFIG.selected_models) - set(MODEL_REGISTRY))
if unknown_models:
    raise ValueError(f'Selected models are not registered: {unknown_models}')

run_framework_self_checks()
suite = unittest.defaultTestLoader.discover(str(PROJECT_DIR / 'tests'))
test_result = unittest.TextTestRunner(verbosity=1).run(suite)
if not test_result.wasSuccessful():
    raise RuntimeError('Pipeline unit tests failed.')
print('Drive project files: PASS')
print('Selected model registry: PASS')
print('Framework self-checks: PASS')
print('Pipeline unit tests: PASS')

.........
----------------------------------------------------------------------
Ran 9 tests in 0.082s

OK


Drive project files: PASS
Selected model registry: PASS
Framework self-checks: PASS
Pipeline unit tests: PASS


## 4. Run or resume

This is safe to rerun. Compatible completed refits load, incomplete work resumes or reruns, pooled files rebuild from refit outputs, and metrics and portfolios refresh.

In [ ]:
# Run one model at a time so pandas never constructs the union of every
# model's lagged feature blocks in memory. Artifacts and resume logic are unchanged.
from dataclasses import replace
import gc
import torch
from src.runner import ExperimentRunner

for model_id in CONFIG.selected_models:
    print(f'\n=== {model_id} ===')
    model_config = replace(CONFIG, selected_models=(model_id,))
    ExperimentRunner(model_config).run()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Restore the full selected-model view for comparison and later cells.
runner = ExperimentRunner(CONFIG)
comparison = runner._cumulative_comparison()
display(comparison)


=== HYBRID_LGBM40_DEEPSET40 ===
Device: cuda

HYBRID_LGBM40_DEEPSET40 [5d043e5c4e15d59b]
  refit 01/26 | test_year=1999 | status=training | train_n=852,623 | validation_n=308,654 | test_n=75,023


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  refit 01/26 | test_year=1999 | status=saved | n_predictions=75,023
  refit 02/26 | test_year=2000 | status=training | train_n=884,456 | validation_n=310,498 | test_n=74,037


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  refit 02/26 | test_year=2000 | status=saved | n_predictions=74,037
  refit 03/26 | test_year=2001 | status=training | train_n=919,072 | validation_n=306,095 | test_n=68,279


...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  refit 25/26 | test_year=2023 | status=saved | n_predictions=52,611
  refit 26/26 | test_year=2024 | status=training | train_n=685,643 | validation_n=205,987 | test_n=48,561


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  refit 26/26 | test_year=2024 | status=saved | n_predictions=48,561


,model_id,model_signature,pooled_oos_r2,n_predictions,diagnostics_version,robust_oos_r2,mean_monthly_rank_ic,stock_count_weighted_mean_rank_ic,rank_ic_std,rank_ic_information_ratio,...,tail_5pct_mean_short_coverage,mean_monthly_return,annualized_return,annualized_volatility,sharpe,t_stat,newey_west_t_stat,max_drawdown,hit_rate,n_months
6,HYBRID_LGBM20_DEEPSET20,e174e09b8fb6c908,0.003777,1323218,post_model_v5_missing_return_stress,0.004731,0.064932,0.069522,0.098261,2.289125,...,0.997274,0.024662,0.295947,0.199933,1.480230,7.547720,5.123272,-0.442996,0.714744,312
17,LGBM_40_LAG2,0edb2824bcbddeea,0.003492,1323218,post_model_v5_missing_return_stress,0.004245,0.055600,0.060667,0.092292,2.086903,...,0.996799,0.024073,0.288878,0.183637,1.573089,8.021210,5.509337,-0.382670,0.759615,312
13,LGBM_20_LAG1,bdec171e3c2c897e,0.003163,1323218,post_model_v5_missing_return_stress,0.003931,0.061383,0.066074,0.101834,2.088063,...,0.996929,0.025064,0.300770,0.196777,1.528481,7.793753,5.645555,-0.313098,0.721154,312
12,LGBM_20,22f55f0c4ee94d78,0.003154,1323218,post_model_v5_missing_return_stress,0.003896,0.062117,0.066662,0.099748,2.157216,...,0.996566,0.023546,0.282546,0.199388,1.417068,7.225658,5.458950,-0.379957,0.682692,312
14,LGBM_20_LAG2,754b8a09606f2ba3,0.003055,1323218,post_model_v5_missing_return_stress,0.003821,0.057368,0.062178,0.101104,1.965581,...,0.997008,0.022830,0.273959,0.199636,1.372292,6.997346,5.133194,-0.378665,0.708333,312
15,LGBM_40,c7e5a69df65fab9c,0.002943,1323218,post_model_v5_missing_return_stress,0.003493,0.052378,0.057046,0.085670,2.117936,...,0.996871,0.023762,0.285139,0.170520,1.672168,8.526417,5.681712,-0.280774,0.733974,312
16,LGBM_40_LAG1,2ae8896afb73218a,0.002880,1323218,post_model_v5_missing_return_stress,0.003419,0.056838,0.061699,0.093240,2.111688,...,0.996810,0.024741,0.296887,0.184110,1.612554,8.222444,5.623970,-0.320538,0.756410,312
0,DEEPSET_20,54f35dd2624edc1f,0.002570,1323218,post_model_v5_missing_return_stress,0.003134,0.061971,0.066433,0.094087,2.281643,...,0.997849,0.022456,0.269475,0.199572,1.350264,6.885023,4.559141,-0.388498,0.679487,312
26,XGBOOST_20,7d503cd960f0625b,0.002333,1323218,post_model_v5_missing_return_stress,0.003086,0.060332,0.064647,0.098346,2.125113,...,0.996599,0.022184,0.266209,0.196761,1.352960,6.898768,5.208277,-0.399388,0.698718,312
7,HYBRID_LGBM40_DEEPSET40,5d043e5c4e15d59b,0.002312,1323218,post_model_v5_missing_return_stress,0.002572,0.058504,0.062603,0.085961,2.357626,...,0.998092,0.027387,0.328643,0.181905,1.806679,9.212292,5.888181,-0.271111,0.750000,312


## 5. Reload saved comparison without training

In [ ]:
import pandas as pd

comparison_path = CONFIG.run_dir / 'model_comparison.csv'
if comparison_path.exists():
    display(pd.read_csv(comparison_path))
else:
    print('No completed model comparison exists yet.')

,model_id,model_signature,pooled_oos_r2,n_predictions,diagnostics_version,robust_oos_r2,mean_monthly_rank_ic,stock_count_weighted_mean_rank_ic,rank_ic_std,rank_ic_information_ratio,...,tail_5pct_mean_short_coverage,mean_monthly_return,annualized_return,annualized_volatility,sharpe,t_stat,newey_west_t_stat,max_drawdown,hit_rate,n_months
0,HYBRID_LGBM20_DEEPSET20,e174e09b8fb6c908,0.003777,1323218,post_model_v5_missing_return_stress,0.004731,0.064932,0.069522,0.098261,2.289125,...,0.997274,0.024662,0.295947,0.199933,1.480230,7.547720,5.123272,-0.442996,0.714744,312
1,LGBM_40_LAG2,0edb2824bcbddeea,0.003492,1323218,post_model_v5_missing_return_stress,0.004245,0.055600,0.060667,0.092292,2.086903,...,0.996799,0.024073,0.288878,0.183637,1.573089,8.021210,5.509337,-0.382670,0.759615,312
2,LGBM_20_LAG1,bdec171e3c2c897e,0.003163,1323218,post_model_v5_missing_return_stress,0.003931,0.061383,0.066074,0.101834,2.088063,...,0.996929,0.025064,0.300770,0.196777,1.528481,7.793753,5.645555,-0.313098,0.721154,312
3,LGBM_20,22f55f0c4ee94d78,0.003154,1323218,post_model_v5_missing_return_stress,0.003896,0.062117,0.066662,0.099748,2.157216,...,0.996566,0.023546,0.282546,0.199388,1.417068,7.225658,5.458950,-0.379957,0.682692,312
4,LGBM_20_LAG2,754b8a09606f2ba3,0.003055,1323218,post_model_v5_missing_return_stress,0.003821,0.057368,0.062178,0.101104,1.965581,...,0.997008,0.022830,0.273959,0.199636,1.372292,6.997346,5.133194,-0.378665,0.708333,312
5,LGBM_40,c7e5a69df65fab9c,0.002943,1323218,post_model_v5_missing_return_stress,0.003493,0.052378,0.057046,0.085670,2.117936,...,0.996871,0.023762,0.285139,0.170520,1.672168,8.526417,5.681712,-0.280774,0.733974,312
6,LGBM_40_LAG1,2ae8896afb73218a,0.002880,1323218,post_model_v5_missing_return_stress,0.003419,0.056838,0.061699,0.093240,2.111688,...,0.996810,0.024741,0.296887,0.184110,1.612554,8.222444,5.623970,-0.320538,0.756410,312
7,DEEPSET_20,54f35dd2624edc1f,0.002570,1323218,post_model_v5_missing_return_stress,0.003134,0.061971,0.066433,0.094087,2.281643,...,0.997849,0.022456,0.269475,0.199572,1.350264,6.885023,4.559141,-0.388498,0.679487,312
8,XGBOOST_20,7d503cd960f0625b,0.002333,1323218,post_model_v5_missing_return_stress,0.003086,0.060332,0.064647,0.098346,2.125113,...,0.996599,0.022184,0.266209,0.196761,1.352960,6.898768,5.208277,-0.399388,0.698718,312
9,HYBRID_LGBM40_DEEPSET40,5d043e5c4e15d59b,0.002312,1323218,post_model_v5_missing_return_stress,0.002572,0.058504,0.062603,0.085961,2.357626,...,0.998092,0.027387,0.328643,0.181905,1.806679,9.212292,5.888181,-0.271111,0.750000,312


## 6. Portfolio implementability robustness

This cached diagnostic reads each pooled prediction file once and never loads model weights. It evaluates the 10% tail portfolio under full/ex-microcap universes, equal/value weighting, fixed proportional transaction-cost scenarios, and the adverse missing-return stress.

In [ ]:
from src.portfolio_robustness import run_portfolio_robustness

portfolio_robustness = run_portfolio_robustness(
    CONFIG.run_dir,
    model_ids=CONFIG.selected_models,
)
display(portfolio_robustness)

HYBRID_LGBM40_DEEPSET40: saved portfolio robustness
HYBRID_LGBM40_DEEPSET40_DYNAMIC: saved portfolio robustness


,universe,weighting,mean_monthly_turnover,annualized_turnover,mean_n_eligible,mean_long_coverage,mean_short_coverage,gross_mean_monthly_return,gross_annualized_return,gross_annualized_volatility,...,missing_return_stress_annualized_return,missing_return_stress_annualized_volatility,missing_return_stress_sharpe,missing_return_stress_t_stat,missing_return_stress_newey_west_t_stat,missing_return_stress_max_drawdown,missing_return_stress_hit_rate,missing_return_stress_n_months,model_id,model_signature
0,EX_MICRO,EQUAL,0.965088,11.581061,2266.413462,0.995135,0.997464,0.015354,0.184247,0.211348,...,0.154324,0.208310,0.740837,3.777544,2.870749,-0.423226,0.621795,312,HYBRID_LGBM40_DEEPSET40,5d043e5c4e15d59b
1,EX_MICRO,VALUE,1.151814,13.821771,2266.413462,0.995135,0.997464,0.012030,0.144365,0.225896,...,0.124343,0.224323,0.554300,2.826389,2.501459,-0.481435,0.589744,312,HYBRID_LGBM40_DEEPSET40,5d043e5c4e15d59b
2,FULL,EQUAL,0.959932,11.519179,4241.083333,0.994484,0.997807,0.027396,0.328749,0.181921,...,0.289180,0.179207,1.613667,8.228118,5.375074,-0.311317,0.727564,312,HYBRID_LGBM40_DEEPSET40,5d043e5c4e15d59b
3,FULL,VALUE,1.158444,13.901333,4241.083333,0.994484,0.997807,0.018419,0.221033,0.238453,...,0.196735,0.237469,0.828470,4.224383,3.182505,-0.422156,0.621795,312,HYBRID_LGBM40_DEEPSET40,5d043e5c4e15d59b
4,EX_MICRO,EQUAL,1.071156,12.853868,2266.413462,0.993344,0.997300,0.014397,0.172758,0.182339,...,0.134728,0.178635,0.754211,3.845734,3.035457,-0.363590,0.608974,312,HYBRID_LGBM40_DEEPSET40_DYNAMIC,a981b2935bfbec2b
5,EX_MICRO,VALUE,1.245285,14.943419,2266.413462,0.993344,0.997300,0.012064,0.144773,0.195772,...,0.115523,0.197458,0.585049,2.983176,2.611097,-0.388546,0.570513,312,HYBRID_LGBM40_DEEPSET40_DYNAMIC,a981b2935bfbec2b
6,FULL,EQUAL,1.049839,12.598066,4241.083333,0.992426,0.997220,0.026709,0.320508,0.164629,...,0.267601,0.159447,1.678307,8.557719,5.660802,-0.279886,0.733974,312,HYBRID_LGBM40_DEEPSET40_DYNAMIC,a981b2935bfbec2b
7,FULL,VALUE,1.278941,15.347289,4241.083333,0.992426,0.997220,0.014491,0.173893,0.208299,...,0.129978,0.210117,0.618600,3.154253,2.836680,-0.441626,0.570513,312,HYBRID_LGBM40_DEEPSET40_DYNAMIC,a981b2935bfbec2b


## 7. Paired model tests

Run this only after the required models have final monthly diagnostic and portfolio-variant files. The default is strict: a missing planned pair raises an error rather than silently producing an incomplete comparison.

In [ ]:
from src.model_comparison import run_paired_model_comparisons

paired_comparison = run_paired_model_comparisons(CONFIG.run_dir, seed=CONFIG.seed)
display(paired_comparison)

,model_a,model_b,difference_definition,mean_monthly_mse_difference,mse_difference_nw_t_stat,mse_difference_p_value,mean_monthly_return_difference,return_difference_nw_t_stat,return_difference_p_value,mean_rank_ic_difference,rank_ic_difference_clustered_t_stat,rank_ic_difference_p_value,sharpe_difference,sharpe_difference_block_bootstrap_ci_low,sharpe_difference_block_bootstrap_ci_high,n_paired_months,mse_difference_holm_p_value,return_difference_holm_p_value,rank_ic_difference_holm_p_value
0,NN4_20,NN2_20,model_a_minus_model_b,0.000085,2.447613,0.014381,-0.006154,-3.173995,0.001504,-0.010651,-3.522066,0.000428,-0.208632,-0.429739,0.025411,312,0.287612,0.031575,0.009420
1,NN4_40,NN2_40,model_a_minus_model_b,0.000042,1.382866,0.166706,-0.000665,-0.309131,0.757222,-0.002481,-0.527294,0.597990,0.020248,-0.340143,0.420391,312,1.000000,1.000000,1.000000
2,NN2_40,NN2_20,model_a_minus_model_b,0.000019,0.696132,0.486346,0.001988,1.214534,0.224544,-0.003416,-1.105543,0.268924,0.311107,-0.012107,0.622037,312,1.000000,1.000000,1.000000
3,NN2_40,MLP_40,model_a_minus_model_b,-0.000007,-0.233351,0.815489,-0.005112,-3.690598,0.000224,-0.008218,-2.641503,0.008254,-0.313106,-0.545911,-0.090419,312,1.000000,0.004922,0.156824
4,NN4_40,NN4_20,model_a_minus_model_b,-0.000025,-0.581042,0.561212,0.007477,2.857202,0.004274,0.004755,0.853160,0.393571,0.539987,0.253958,0.891892,312,1.000000,0.085479,1.000000
5,NN4_20,NN3_20,model_a_minus_model_b,0.000067,1.733532,0.083001,-0.003217,-1.711090,0.087064,-0.006259,-1.893869,0.058242,-0.137604,-0.449229,0.158513,312,1.000000,1.000000,0.890419
6,LGBM_40,LGBM_20,model_a_minus_model_b,0.000010,0.560382,0.575219,0.000223,0.125896,0.899814,-0.009738,-2.569207,0.010193,0.255306,0.012214,0.530322,312,1.000000,1.000000,0.183477
7,LGBM_40,LGBM_40_LAG1,model_a_minus_model_b,-0.000001,-0.089146,0.928966,-0.000980,-0.859211,0.390224,-0.004460,-1.831109,0.067084,0.059203,-0.121213,0.243568,312,1.000000,1.000000,0.939179
8,LGBM_20_LAG1,LGBM_20_LAG2,model_a_minus_model_b,-0.000003,-0.534848,0.592755,0.002236,2.443242,0.014556,0.004015,1.926454,0.054048,0.156300,-0.021081,0.330286,312,1.000000,0.276563,0.890419
9,LGBM_40,LGBM_40_LAG2,model_a_minus_model_b,0.000022,1.397113,0.162379,-0.000308,-0.265636,0.790519,-0.003222,-1.641274,0.100741,0.099239,-0.134326,0.332608,312,1.000000,1.000000,1.000000


## 8. Final completion checks and frozen outputs

In [ ]:
# Recreate the lightweight runner in case the kernel was restarted or this
# cell is run independently. This does not call runner.run() or train models.
from src.runner import ExperimentRunner
runner = ExperimentRunner(CONFIG)
final_comparison = runner._cumulative_comparison()
expected_models = set(CONFIG.selected_models)
completed_models = set(final_comparison.loc[
    final_comparison['diagnostics_version'].eq(runner.DIAGNOSTICS_VERSION), 'model_id'
])
missing_current = sorted(expected_models - completed_models)
if missing_current:
    raise RuntimeError(f'Models missing current diagnostics: {missing_current}')
expected_oos_months = 12 * (
    CONFIG.universe.end_year - CONFIG.universe.start_year
    - CONFIG.windows.train_years - CONFIG.windows.validation_years + 1
)
selected_comparison = final_comparison.set_index('model_id').loc[list(expected_models)]
if selected_comparison['n_months'].ne(expected_oos_months).any():
    raise RuntimeError('At least one model does not cover the complete OOS portfolio calendar.')
if (selected_comparison['n_signal_months'] + selected_comparison['n_no_signal_months']).ne(expected_oos_months).any():
    raise RuntimeError('At least one model has an incomplete signal-availability accounting.')
required_columns = [
    'robust_oos_r2', 'mean_monthly_rank_ic', 'n_no_signal_months',
    'mean_monthly_calibration_slope', 'tail_5pct_sharpe',
    'tail_10pct_sharpe', 'tail_20pct_sharpe', 'rank_weighted_sharpe',
    'tail_10pct_missing_return_stress_annualized_return',
]
missing_values = final_comparison.set_index('model_id').loc[list(expected_models), required_columns].isna()
if missing_values.any().any():
    print('Legitimate undefined diagnostics require review:')
    display(missing_values[missing_values.any(axis=1)])
else:
    print('All selected models have current complete diagnostics.')
expected_robustness_rows = 4 * len(expected_models)
if len(portfolio_robustness) != expected_robustness_rows:
    raise RuntimeError(
        f'Expected {expected_robustness_rows} robustness rows, got {len(portfolio_robustness)}.'
    )
from src.model_comparison import DEFAULT_MODEL_PAIRS
if len(paired_comparison) != len(DEFAULT_MODEL_PAIRS):
    raise RuntimeError('Paired model comparison is incomplete.')
print('Portfolio robustness and paired model comparisons are complete.')

All selected models have current complete diagnostics.
Portfolio robustness and paired model comparisons are complete.


## Adding a model later

Add a `ModelSpec` and trainer in `src/models.py`, then add its ID to `selected_models`. Existing completed model signatures remain unchanged. Add a focused self-check for every new architecture before starting a full rolling run.